# Stage 1 Final Overview

Decision-oriented overview for Stage 1 architecture search on `target_log_return_20m` with `price_hmm_n4` features.

Source run:

`s3://binance-data-downloader/dataset_target_20/with_price_hmm_n4/stage1_architecture_search/20260709_204850/`

## Executive Summary

- Final `stage1_results.parquet` exists in S3.
- Completed models: **99**.
- Original queue size in `run_config.json`: **135**.
- Completed families/losses: CatBoost RMSE, CatBoost RMSEWithUncertainty, LightGBM RMSE.
- Missing from the original plan: **CatBoost MultiQuantile, 36 models**.
- Best Stage 1 quality is from **CatBoost RMSEWithUncertainty**.
- Best RMSE: **0.00716098**, model `catboost_uncertainty_002_d4_lr0.03_l210`.
- Best direction accuracy at 0.25% threshold: **0.518447**, model `catboost_uncertainty_012_d6_lr0.03_l230`.
- LightGBM is much faster but materially weaker by RMSE in this run.

Main pruning decision:

- For CatBoost uncertainty, keep `depth=4/6`, mostly `learning_rate=0.03/0.05`; `depth=8/10` are not worth Stage 2 as primary RMSE candidates.
- Use `depth=6` variants only when direction accuracy is prioritized.
- Treat `learning_rate=0.1` as a low-priority edge case, not the center of Stage 2.
- Run MultiQuantile separately only if probabilistic intervals are required; it was not part of the completed final table.

## Setup

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "build_price_feature_day.py").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_price_feature_day import make_s3_client
from analysis.stage1_model_search.stage1_analysis_tools import (
    add_composite_score,
    early_stop_curve,
    hyperparameter_importance,
    load_stage1_results,
    parameter_columns,
    parameter_correlations,
    parameter_summary,
    pareto_frontier,
    plot_early_stop,
    plot_heatmap,
    plot_metric_distributions,
    plot_pareto,
    read_parquet,
    run_overview,
    stability_table,
    top_tables,
)

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 220)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
BUCKET = "binance-data-downloader"
DATASET_PREFIX = "dataset_target_20/with_price_hmm_n4"
RESULTS_SUBDIR = "stage1_architecture_search"
RUN_ID = "latest"

s3 = make_s3_client()
loaded = load_stage1_results(
    s3=s3,
    bucket=BUCKET,
    dataset_prefix=DATASET_PREFIX,
    results_subdir=RESULTS_SUBDIR,
    run_id=RUN_ID,
)
results = add_composite_score(loaded.results)
run_config = loaded.run_config

print(loaded.active_s3_uri)
print(f"loaded_from_final_table={loaded.loaded_from_final_table}")
print(f"rows={len(results):,}")
run_config

## 1. Completion Check

In [ ]:
overview = run_overview(results, run_config)
overview

In [ ]:
completed_counts = results.groupby(["model_family", "loss_function"]).size().rename("completed").reset_index()
completed_counts

Interpretation: the final table is complete for the **restarted non-MultiQuantile subset**, but not for the original 135-job plan. MultiQuantile should be treated as not evaluated in this Stage 1 conclusion.

## 2. Family-Level Result

In [ ]:
family_summary = (
    results.groupby(["model_family", "loss_function"], dropna=False)
    .agg(
        jobs=("job_id", "count"),
        best_RMSE=("RMSE", "min"),
        median_RMSE=("RMSE", "median"),
        best_MAE=("MAE", "min"),
        best_DA_025=("Direction_Accuracy_0.25%", "max"),
        median_train_time=("train_time", "median"),
        median_model_size_mb=("model_size_mb", "median"),
    )
    .reset_index()
    .sort_values("best_RMSE")
)
family_summary

In [ ]:
plot_metric_distributions(results)

## 3. Leaderboards

In [ ]:
leaderboards = top_tables(results, n=10)
for name, table in leaderboards.items():
    print(f"\n{name}")
    display(table)

RMSE winners are strongly concentrated in CatBoost RMSEWithUncertainty. Direction-accuracy winners shift toward `depth=6/8`, but the RMSE cost and train-time cost become visible quickly.

## 4. CatBoost Hyperparameter Pruning

In [ ]:
catboost = results.loc[results["model_family"].eq("catboost")].copy()
catboost_uncertainty = catboost.loc[catboost["loss_function"].eq("RMSEWithUncertainty")].copy()

for param in ["param_depth", "param_learning_rate", "param_l2_leaf_reg"]:
    print(f"\n{param} across CatBoost")
    display(parameter_summary(catboost, param))

for param in ["param_depth", "param_learning_rate", "param_l2_leaf_reg"]:
    print(f"\n{param} for CatBoost RMSEWithUncertainty")
    display(parameter_summary(catboost_uncertainty, param))

In [ ]:
for row_param, col_param in [
    ("param_depth", "param_learning_rate"),
    ("param_depth", "param_l2_leaf_reg"),
    ("param_learning_rate", "param_l2_leaf_reg"),
]:
    pivot = catboost_uncertainty.pivot_table(index=row_param, columns=col_param, values="RMSE", aggfunc="median")
    display(pivot)
    plot_heatmap(pivot, title=f"CatBoost uncertainty median RMSE: {row_param} x {col_param}")

CatBoost pruning decision:

- Exclude `depth=10` from Stage 2.
- Exclude `depth=8` from RMSE-centered Stage 2, unless direction accuracy is the priority.
- Center Stage 2 around `depth=4/6`.
- Center learning rate around `0.03/0.05`.
- Keep all tested `l2_leaf_reg` values as seeds; differences are small, so regularization search is more useful than a hard l2 cutoff.

## 5. LightGBM Hyperparameter Pruning

In [ ]:
lightgbm = results.loc[results["model_family"].eq("lightgbm")].copy()
for param in ["param_num_leaves", "param_learning_rate", "param_max_depth"]:
    print(f"\n{param}")
    display(parameter_summary(lightgbm, param))

for row_param, col_param in [
    ("param_num_leaves", "param_learning_rate"),
    ("param_max_depth", "param_learning_rate"),
    ("param_max_depth", "param_num_leaves"),
]:
    pivot = lightgbm.pivot_table(index=row_param, columns=col_param, values="RMSE", aggfunc="median")
    display(pivot)
    plot_heatmap(pivot, title=f"LightGBM median RMSE: {row_param} x {col_param}")

LightGBM pruning decision:

- Keep LightGBM as a fast baseline and possible speed-quality Pareto candidate.
- For quality, LightGBM trails CatBoost uncertainty.
- Exclude `learning_rate=0.1`.
- Prefer `num_leaves=31`, then `63`; `127` is not attractive as a center.
- `max_depth=10` and `-1` are both viable; `6` is weaker but still cheap.

## 6. Cost-Quality Trade-Off

In [ ]:
frontier = pareto_frontier(results, quality_metric="RMSE", cost_metric="train_time")
pareto_columns = ["job_id", "model_family", "loss_function", "RMSE", "MAE", "Direction_Accuracy_0.25%", "train_time", "model_size_mb", "params"]
display(frontier.loc[:, pareto_columns])
plot_pareto(results, frontier, quality_metric="RMSE", cost_metric="train_time")

In [ ]:
stability_table(catboost_uncertainty, metric="RMSE", sizes=(5, 10, 20, 30))

Top-10 CatBoost uncertainty RMSE spread is under 0.1% of the best result. This suggests Stage 2 should not chase deeper models; it should test regularization around the stable region.

## 7. Hyperparameter Importance

In [ ]:
for label, frame in {
    "CatBoost uncertainty": catboost_uncertainty,
    "LightGBM RMSE": lightgbm,
}.items():
    importance, model, matrix = hyperparameter_importance(frame, metric="RMSE", param_cols=parameter_columns(frame))
    print(f"\n{label}")
    display(importance)

parameter_correlations(results)

## 8. HMM Feature Importance Check

In [ ]:
def load_feature_importance(row: pd.Series) -> pd.DataFrame:
    key = row.get("feature_importance_key")
    if not isinstance(key, str) or not key:
        return pd.DataFrame()
    frame = read_parquet(s3, BUCKET, key)
    if frame.empty:
        return frame
    frame = frame.copy()
    frame["job_id"] = row["job_id"]
    frame["model_family"] = row["model_family"]
    frame["loss_function"] = row["loss_function"]
    frame["RMSE"] = row["RMSE"]
    return frame

top_for_importance = results.sort_values(["RMSE", "MAE", "job_id"]).head(10)
feature_importance = pd.concat(
    [load_feature_importance(row) for _, row in top_for_importance.iterrows()],
    ignore_index=True,
) if len(top_for_importance) else pd.DataFrame()

if feature_importance.empty:
    print("No feature importance available")
else:
    feature_importance["importance_abs"] = feature_importance["importance"].abs()
    aggregate_importance = (
        feature_importance.groupby("feature", as_index=False)
        .agg(mean_abs_importance=("importance_abs", "mean"), jobs=("job_id", "nunique"))
        .sort_values("mean_abs_importance", ascending=False)
    )
    display(aggregate_importance.head(30))
    display(aggregate_importance.loc[aggregate_importance["feature"].str.startswith("price_hmm_n4_")])

## 9. Early-Stop Diagnostic

In [ ]:
curve = early_stop_curve(results, metric="RMSE")
display(curve.head(20))
display(curve.tail(20))
plot_early_stop(curve, metric="RMSE")

## 10. Recommended Stage 2

Recommended main Stage 2 queue: **CatBoost RMSEWithUncertainty regularization search** around stable Stage 1 seeds.

Use this as the main queue because it wins RMSE and direction accuracy while keeping a stable top region.

In [ ]:
ARCHITECTURE_SEEDS_STAGE2_MAIN = [
    {"depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 3},
    {"depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 30},
    {"depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 10},
    {"depth": 4, "learning_rate": 0.05, "l2_leaf_reg": 30},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 3},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 30},
]

GRID_STAGE2_REGULARIZATION = {
    "bagging_temperature": [0, 1, 3, 5],
    "rsm": [0.7, 0.85, 1.0],
    "random_strength": [0, 1, 2],
}

stage2_main_jobs = len(ARCHITECTURE_SEEDS_STAGE2_MAIN)
for values in GRID_STAGE2_REGULARIZATION.values():
    stage2_main_jobs *= len(values)

stage2_main_jobs

Compact alternative if time is limited:

In [ ]:
ARCHITECTURE_SEEDS_STAGE2_FAST = [
    {"depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"depth": 4, "learning_rate": 0.03, "l2_leaf_reg": 30},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 10},
    {"depth": 6, "learning_rate": 0.03, "l2_leaf_reg": 30},
]

stage2_fast_jobs = len(ARCHITECTURE_SEEDS_STAGE2_FAST)
for values in GRID_STAGE2_REGULARIZATION.values():
    stage2_fast_jobs *= len(values)

stage2_fast_jobs

Secondary queues:

- CatBoost RMSE can be used as a point-forecast baseline, but it does not beat uncertainty loss.
- LightGBM RMSE is useful as a fast baseline or Pareto-speed model, centered on `num_leaves=31`, `learning_rate=0.03/0.05`, `max_depth=10/-1`.
- CatBoost MultiQuantile should be launched as a separate Stage 1 recovery run before including it in Stage 2 decisions.